# Fig. — min/mean-SINR CDF + SCNR CDF at $\gamma^\star$

Loader/summary helpers come from `build_fig_cdf.py`; **`build_figure` is inlined
below as an editable cell** so you can override style without editing the module.

- **(a)** SINR CDF at γ\* — `SINR_METRIC='min'` (worst-user) or `'mean'` (across users).
- **(b)** SCNR CDF at γ\*.
- Legend annotation = **moderate-outage** feasibility at γ\*: `FEASIBILITY='served'`
  (fraction of trials with ≥η of users ≥ γ\*), `'outage'` (per-user Pr(SINR<γ\*)), or
  `'strict'` (legacy all-or-nothing). Default is `served`.

**Compatible experiments:** `sinr_cdf`, `scnr_cdf` (`kind=='single'`).

In [ ]:
import sys, logging
from pathlib import Path
import numpy as np
import matplotlib
import matplotlib.pyplot as plt

FIG_DIR = Path.cwd()
if str(FIG_DIR) not in sys.path:
    sys.path.insert(0, str(FIG_DIR))

import build_fig_cdf as B   # loader + helpers (single source of truth)
from cordis.plotting import apply_paper_style, figsize, plot_cdf, save_figure
from cordis.plotting.style import ALGORITHM_STYLE   # tweak here to restyle

USE_TEX = True   # set False on a node without pdflatex
apply_paper_style()
logging.basicConfig(level=logging.INFO, format='%(levelname)-7s %(message)s')

## 1. Load the CDF runs

In [ ]:
SINR_DIR = None   # e.g. 'results/exp_sinr_cdf/array_12345_aggregated'
SCNR_DIR = None
sinr_result, sinr_dir = B.load_cdf(SINR_DIR, experiment='sinr_cdf')
try:
    scnr_result, scnr_dir = B.load_cdf(SCNR_DIR, experiment='scnr_cdf')
except FileNotFoundError:
    print('No scnr_cdf run; reusing the sinr_cdf run for panel (b).')
    scnr_result, scnr_dir = sinr_result, sinr_dir
print('SINR run:', sinr_dir); print('SCNR run:', scnr_dir)

## 2. Knobs: operating point, SINR metric, feasibility definition

In [ ]:
GAMMA_DB    = None        # None -> auto-detect γ* from metadata
SINR_METRIC = 'min'       # 'min' (worst-user) or 'mean' (across users)
FEASIBILITY = 'served'    # 'served' (moderate) | 'outage' (moderate) | 'strict' (legacy)
ETA         = 0.9         # coverage fraction η for 'served'
ONLY        = ['CORDIS-Split', r'LR-MMSE ($\rho$=0.5)', 'Global-ZF']
SCNR_METRIC = "weighted_sum_scnr_db"        # "min_scnr_db", "weighted_sum_scnr_db", "mean_scnr_db"

gamma_db, detected = (GAMMA_DB, True) if GAMMA_DB is not None \
    else B._detect_gamma_db(sinr_result)
only = [n for n in ONLY if n in sinr_result.sim_result.names] \
    or B._present_algorithms(sinr_result, B.PREFERRED)
scnr_metric = SCNR_METRIC or B._select_scnr_metric(scnr_result, only=only)
sinr_metric_name, sinr_label = B._resolve_sinr_metric(SINR_METRIC)
print(f'γ*={gamma_db:g} dB (detected={detected})\nSINR={sinr_label}\n'
      f'feasibility={FEASIBILITY}; SCNR={scnr_metric}')

## 3. Numeric summary

Reports the chosen SINR metric (p50/p05), the moderate-outage feasibility scalar
at γ\*, the (1−ε)-likely per-user SINR, and median SCNR.

In [ ]:
summary = B.collect_summary(sinr_result, scnr_result, gamma_db, only, scnr_metric,
                            sinr_metric=sinr_metric_name, feasibility=FEASIBILITY, eta=ETA)
B._print_summary(summary, gamma_db, scnr_metric,
                 sinr_label=sinr_label, feasibility=FEASIBILITY, eta=ETA)

## 4. `build_figure` — editable copy

The **exact** function from `build_fig_cdf.py`, inlined so you can edit it and re-run.
Mutate `ALGORITHM_STYLE` in the setup cell to restyle without editing the body.

In [ ]:
# Bind the module-level names the function body references.
from typing import Optional, Sequence   # the inlined signature uses these
PREFERRED            = B.PREFERRED
FEASIBILITY_DEFAULT  = B.FEASIBILITY_DEFAULT
ETA_DEFAULT          = B.ETA_DEFAULT
_SCNR_XLABEL         = B._SCNR_XLABEL
_present_algorithms  = B._present_algorithms
_select_scnr_metric  = B._select_scnr_metric
_resolve_sinr_metric = B._resolve_sinr_metric
_feasibility_value   = B._feasibility_value
_feas_legend_token   = B._feas_legend_token
logger               = logging.getLogger('fig_cdf.nb')

In [ ]:
def build_figure(sinr_result,
                 scnr_result,
                 *,
                 gamma_db: float,
                 only: Optional[Sequence[str]] = None,
                 scnr_metric: Optional[str] = None,
                 sinr_metric: str = "min",
                 feasibility: str = FEASIBILITY_DEFAULT,
                 eta: float = ETA_DEFAULT,
                 use_tex: bool = True):
    """Assemble the two-panel CDF figure and return ``(fig, only, scnr_metric)``.

    Panel (a) is the CDF of the chosen per-trial SINR metric (``sinr_metric``:
    'min' = worst-user, 'mean' = across-user) with γ\\* marked.  Each legend
    entry is annotated with the moderate-outage feasibility scalar at γ\\*
    (``feasibility``: 'served' = served-trial rate ≥η users, 'outage' = per-user
    Pr(SINR<γ\\*), 'strict' = legacy all-or-nothing).
    """
    import matplotlib
    if use_tex is False:
        matplotlib.rcParams["text.usetex"] = False
    import matplotlib.pyplot as plt
    from cordis.plotting import apply_paper_style, figsize, plot_cdf

    apply_paper_style()
    if use_tex is False:
        # apply_paper_style may flip usetex back on; force it off for nodes
        # without a TeX install.
        matplotlib.rcParams["text.usetex"] = False
    using_tex = bool(matplotlib.rcParams.get("text.usetex", False))

    # Define the IEEE standard font size
    IEEE_FONT_SIZE = 8
    plt.rcParams.update({
        # Base font size (affects default text and titles)
        "font.size": IEEE_FONT_SIZE,
        # Axis labels (e.g., 'ADMM iteration', 'min-SINR [dB]')
        "axes.labelsize": IEEE_FONT_SIZE+2,
        # Axis tick labels (the numbers on the axes)
        "xtick.labelsize": IEEE_FONT_SIZE,
        "ytick.labelsize": IEEE_FONT_SIZE,
        # Legend font size
        "legend.fontsize": IEEE_FONT_SIZE,
        # Legend title font size (if you ever use legend titles)
        "legend.title_fontsize": IEEE_FONT_SIZE,
        # Figure titles (if you ever use fig.suptitle)
        "figure.titlesize": IEEE_FONT_SIZE,
        # Subplot titles (e.g., if you switch SET_TITLE to True)
        "axes.titlesize": IEEE_FONT_SIZE,
    })

    sinr_metric_name, sinr_label = _resolve_sinr_metric(sinr_metric)
    if sinr_label == "mean-SINR": sinr_label = "mean-user SINR"
    if sinr_label == "min-SINR": sinr_label = "worst-user SINR"

    only = list(only) if only else _present_algorithms(sinr_result, PREFERRED)
    if scnr_metric is None:
        scnr_metric = _select_scnr_metric(scnr_result, only=only)

    have_scnr = scnr_metric is not None
    ncols = 2 if have_scnr else 1
    fig, axes = plt.subplots(
        1, ncols,
        figsize=figsize(width="double" if have_scnr else "single",
                        aspect=2.4 if have_scnr else (3.5 / 2.6)),
    )
    axes = np.atleast_1d(axes)

    # ── Panel (a): SINR CDF at γ* ───────────────────────────────────
    ax_a = axes[0]
    plot_cdf(
        sinr_result.sim_result,
        metric=sinr_metric_name,
        ax=ax_a,
        xlabel=rf"{sinr_label} [dB]",
        only=only,
        gamma_db=gamma_db,            # vertical γ* line
        annotate_infeasibility=False,  # we annotate moderate-outage below
        legend_loc="lower right",     # any CDF is empty in the lower-right corner
    )
    ax_a.set_ylim(0.0, 1.0)
    ax_a.set_xlim(-75, 40.0)
    # ax_a.set_title(rf"(a) {sinr_label} CDF at $\gamma^\star$")

    # Re-annotate the legend with the moderate-outage feasibility scalar.
    sinr_res = sinr_result.sim_result.algorithm_results
    handles, labels = ax_a.get_legend_handles_labels()
    new_labels = []
    for h, lab in zip(handles, labels):
        ar = sinr_res.get(lab)
        token = None
        if ar is not None:
            fv = _feasibility_value(ar, gamma_db, feasibility, eta, sinr_metric_name)
            if fv is not None:
                token = _feas_legend_token(fv[0], fv[1], using_tex)
        if lab == r'LR-MMSE ($\rho$=0.5)':
            new_labels.append(lab)
        else:
            new_labels.append(f"{lab} ({token})" if token else lab)
    if handles:
        ax_a.legend(handles, new_labels, loc="best")

    # ── Panel (b): SCNR CDF at γ* ───────────────────────────────────
    if have_scnr:
        ax_b = axes[1]
        plot_cdf(
            scnr_result.sim_result,
            metric=scnr_metric,
            ax=ax_b,
            xlabel=_SCNR_XLABEL.get(scnr_metric, scnr_metric),
            only=only,
            legend_loc="best",
        )
        ax_b.set_ylim(0.0, 1.0)
        # ax_b.set_title(r"(b) SCNR CDF at $\gamma^\star$")

    fig.tight_layout()
    return fig, only, scnr_metric


## 5. Render

In [ ]:
fig, used_only, used_scnr_metric = build_figure(
    sinr_result, scnr_result, gamma_db=gamma_db, only=only,
    scnr_metric='min_scnr_db', sinr_metric='min',
    feasibility='outage', eta=0.85, use_tex=USE_TEX,
)
# for scnr_metric: ["min_scnr_db", "weighted_sum_scnr_db", "mean_scnr_db"]
# for sinr_metric: ["min", "mean"]
plt.show()

## 6. Save to `paper/figures/`

In [ ]:
tag = '_split'
out_stem = FIG_DIR.parents[1] / 'figures' / str('fig_cdf' + tag)
paths = save_figure(
    fig, out_stem, formats=('pdf', 'png'),
    metadata={
        'Figure': 'fig_cdf', 'GammaStarDB': f'{gamma_db:g}',
        'SinrMetric': sinr_metric_name,
        'Feasibility': (f'{FEASIBILITY}(eta={ETA:g})' if FEASIBILITY=='served' else FEASIBILITY),
        'Algorithms': ', '.join(used_only), 'ScnrMetric': str(used_scnr_metric),
        'SinrRun': sinr_dir.name, 'ScnrRun': scnr_dir.name,
    },
)
for p in paths:
    print('wrote', p)